# 事前準備：共通コードの実行
* このノートブックに接続したら，まずは以下の2つの共通コード（コードAとコードB）を実行する
* これらの共通コードを実行しないと，それ以降のコードが実行できないので注意する
* また，コードAとコードBは，ノートブックに接続するたび毎回実行すること（ノートブックに接続中は，何度も実行する必要はない）
* 共通コードの詳細についての説明は割愛する（簡単な説明は第2回の「[サンプルノートブック02](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook02.ipynb)」を参照）

In [ ]:
# コードA：日本語化ライブラリ導入
! pip install japanize-matplotlib | tail -n 1

In [ ]:
# コードB：共通事前処理

# B1:余分なワーニングを非表示にする
import warnings
warnings.filterwarnings('ignore')

# 必要ライブラリのimport
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib # matplotlib日本語化対応
import seaborn as sns

# B2:データフレーム表示用関数
from IPython.display import display

# B3:表示オプション調整
np.set_printoptions(suppress = True, precision = 3) #numpyの浮動小数点の表示精度
pd.options.display.float_format = '{:.3f}'.format #pandasでの浮動小数点の表示精度
pd.set_option('display.max_columns', None) #データフレームですべての列データを表示

# B4:グラフのデフォルトフォント指定
plt.rcParams['font.size'] = 14

# 乱数の種
random_seed = 0

## データファイルのダウンロード（コードC）
* 実習を始める前に，今回用いるデータ（csvファイル）を Colab にダウンロードする
* 共通コードと同様に，以下のコードC もノートブックに接続するたび毎回実行すること（ノートブックに接続中は，何度も実行する必要はない）
* 接続が切れると，ファイルは削除されるので注意する
* コードの内容を理解する必要はない

In [ ]:
# コードC：データのダウンロード
!curl -L -o KvsT.csv https://raw.githubusercontent.com/yoshida-nu/lecture_public/main/datascience/resources/03/KvsT.csv
!curl -L -o iris.csv https://raw.githubusercontent.com/yoshida-nu/lecture_public/main/datascience/resources/03/iris.csv

# DataFrameの操作
* ここでは，今回の実習で主に利用する DataFrame の操作を説明する
* 説明が省略されている（説明済みの）部分については，「[第2回サンプルノートブック](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook02.ipynb)」，「[ノートブック：SeriesとDataFrameの基本操作](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook_se_df.ipynb)」，及び教科書第4章・付録Cを適宜参照する

## 列名を指定して列データを抽出
* DataFrame名の後ろに列名のリストを角括弧`[ ]`で囲って記述することで任意の列データを抽出できる
* 以下のコードでは，列名が「Python」「ML」の2列の DataFrame から ML列を抽出している
* 列名を要素とするリストで，抽出する列名を指定する
* 列データ抽出の書式： `df[list]`
  * `df`は任意の DataFrame
  * `list`は抽出する列名（文字列）を要素とするリスト

In [ ]:
score_df = pd.DataFrame([[90, 70], [70, 80], [70, 80], [85, 70]],
                        index = ['工藤', '浅木', '松田', '福田'],
                        columns = ['Python', 'ML'])
cols = ['ML'] #列名「ML」を指定
column = score_df[cols] #ML列を抽出
display(column) #抽出したML列を表示

## 先頭・末尾の数行を返すメソッド
* `head`メソッドと`tail`メソッドで，DataFrameの先頭・末尾の数行を取り出すことができる
* 先頭の`n`行（`n`は整数）を取り出す： `df.head(n)`
* 末尾の`n`行（`n`は整数）を取り出す： `df.tail(n)`

In [ ]:
score_df = pd.DataFrame([[90, 70], [70, 80], [70, 80], [85, 70]],
                        index = ['工藤', '浅木', '松田', '福田'],
                        columns = ['Python', 'ML'])
display(score_df) #DataFrame全体を表示
print('----------------------------------')
display(score_df.head(1)) #先頭1行を表示
print('----------------------------------')
display(score_df.tail(2)) #末尾2行を表示

# 機械学習によるデータ分析の簡単な例
* ここでは，簡単な例を用いて，機械学習によるデータ分析プロセスの流れを把握していく

## データの読み込みと確認
* 例として，csvファイル「KvsT.csv」を用いる
* このデータは，機械学習によるデータ分析体験用の人工データで，以降は「派閥データ」と呼ぶ
* 派閥データは，身長，体重，年代，派閥（きのこ or たけのこ）からなる19個（19人分のデータ）のデータ
* ここでは，身長，体重，年代を説明変数とし，派閥を目的変数とする ⇒ 身長，体重，年代から派閥を予測する
* よって，ここでの予測は分類による予測（教師あり学習）となる
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `display`関数を使ってデータ（`df`の内容）を表示

In [ ]:
# データの読み込み
df = pd.read_csv('KvsT.csv')
display(df)

## 実習： データの前処理
* ここでは教師あり学習を行うので，列データを説明変数と目的変数（正解データ）に分割する
  * 説明変数: 身長，体重，年代
  * 目的変数: 派閥
* それぞれ対応する列名を指定して抽出すればよい
  
**［実習内容］**
* 以下の「以下のコードの処理内容」に従って，コードを完成させる
* 3～6行目に適切なコードを記述する
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* **3行目**: 説明変数に対応する列名を要素とするリストを変数`x_cols`に代入
* **4行目**: 目的変数に対応する列名を要素とするリストを変数`t_col`に代入
* **5行目**: DataFrame `df` から説明変数の列だけ取り出し変数`x`に代入
* **6行目**: DataFrame `df` から目的変数の列だけ取り出し変数`t`に代入
* 7行目: `print`関数で区切り線「========= x =========」を表示
* 8行目: `display`関数で変数`x`（説明変数）の先頭3行を表示
* 9行目: `print`関数で区切り線「========= t =========」を表示
* 10行目: `display`関数で変数`t`（目的変数）の先頭3行を表示

**［実行結果］**

  <img src="./fig/exercise_KvsT_Data_splitting.jpg" width="150">

In [ ]:
# データの分割
df = pd.read_csv('KvsT.csv')




print('========= x =========')
display(x.head(3))
print('========= t =========')
display(t.head(3))

## 機械学習のためのライブラリ
* 本講義では，scikit-learn（サイキットラーン）という外部ライブラリを用いて，機械学習を実践していく
* scikit-learnとは，Pythonで使用できる機械学習ライブラリ（複数のモジュールの集まり）の一つ
* scikit-learnを利用することで様々な機械学習を行うことが可能
* Pythonでscikit-learnを利用する場合には「`sklearn`」と記述する
* `sklearn`には機械学習のタイプに応じて様々なモジュールが用意されている
* 各モジュールの中には，機械学習を行うためのクラスが複数用意されている
* これらのクラスからオブジェクトを生成することで，機械学習が実践できる
* オブジェクトの中にモデルの情報が含まれている

## オブジェクト

### オブジェクトとメソッド
* Pythonではすべての値がオブジェクト
* オブジェクトはあるクラスに属している
* 変数はオブジェクトの入れ物（オブジェクトに名前を付けている）
* クラスには固有のメソッドが定義されている
* メソッドは，そのクラスに属するすべてのオブジェクトが使える
* メソッド＝オブジェクトに対する何らかの処理



### オブジェクトと属性
* クラスにはメソッドの他にも，固有の属性が定義されている
* メソッドは関数のような処理と解釈できるが，属性は変数のような値の入れ物（あるいは値につけられた名前）と解釈できる
* イメージ例: 「人間」クラスのオブジェクト
  * 「歩く」「話す」などがメソッドに対応
  * 「名前」「年齢」「職業」などが属性に対応
* 同じクラスでも異なるオブジェクトであれば異なる属性の値を持つこともある
* 属性の値は変更されることもある


## はじめての機械学習

### 分類木
* ここでは基本的な教師あり学習の一つである分類木を利用する（分類木の詳細は後述）
* 教科書では「決定木」と呼んでいるが，正確には「分類を目的とした決定木」なので，本講義では「分類木」と呼ぶ
* scikit-learnの`tree`モジュールを読み込むことで分類木モデルが利用できる
* `tree`モジュールには「木」で表現できる様々なモデルを扱うためのクラスが複数用意されている
* モデルの学習をするためには，対応するクラスのオブジェクトをまずは生成する必要がある
* 分類木の場合は，`tree`モジュールにおける`DecisionTreeClassifier`クラスのオブジェクトを生成する
  * [DecisionTreeClassifierクラスの公式ドキュメント](https://scikit-learn.org/1.6/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
* オブジェクト生成の書式: `クラス名(引数)`
  * ただし，事前にクラスをインポートしておく必要がある
* 一般に生成したオブジェクトを変数に代入して利用する

### 分類木モデルの学習準備
* `sklearn`（ライブラリ）の`tree`モジュール内の`DecisionTreeClassifier`クラスをインポート
  * 書式： `from sklearn.tree import DecisionTreeClassifier`
* `DecisionTreeClassifier`クラスのオブジェクトを生成する
  * 書式： `DecisionTreeClassifier(random_state=random_seed)`
  * 引数の`random_state`は，学習に乱数を用いる場合に指定できる引数
  * `random_state`で乱数の種（シード）を指定する
  * 乱数には無数のパターンが用意されており，それらパターンの番号のことを乱数の種と呼ぶ
  * 少々乱暴な説明ではあるが，本講義においてはこの解釈で十分
  * 乱数の種を指定することで常に同じ結果が得られる
* 生成したオブジェクトは変数に代入して取り扱う

### 分類木モデルの学習
* オブジェクトを生成したら，分類木モデルの学習を実行する
* 分類木モデルの学習は，`fit`メソッドを利用して実行する
* 他の多くのモデルの学習でも`fit`メソッドを用いる
* `fit`メソッドの引数には，説明変数と目的変数を指定する
* モデルの学習の書式: `変数.fit(X=[説明変数], y=[目的変数])`
  * 変数名はオブジェクト（今回は`DecisionTreeClassifier`クラスのオブジェクト）を代入した変数の名前
  * 「`X=`」で説明変数，「`y=`」で目的変数を指定する
* 学習済みモデルの情報は，`fit`メソッドを適用後のオブジェクトの属性から参照できる
* 同様にモデルの評価のために必要な情報も属性から参照できる
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: 説明変数の列名を要素とするリスト`['身長', '体重', '年代']`を変数`x_cols`に代入
* 4行目: 目的変数の列名（リスト）`['rent_price']`を変数`t_col`に代入
* 5行目: DataFrame `df` から,`df[x_cols]`で，説明変数の列だけ取り出し変数`x`に代入
* 6行目: DataFrame `df` から,`df[t_col]`で目的変数の列だけ取り出し変数`t`に代入
* 7行目: `sklearn` (scikit-learn) の`tree`モジュール内の`DecisionTreeClassifier`クラスをインポート
* 8行目: `DecisionTreeClassifier(random_state=random_seed)`で，分類木モデルの学習を行うためのオブジェクトを`DecisionTreeClassifier`クラスから生成し，変数`model`に代入
  * `random_state`で乱数の種（シード）を指定する
  *  ここでは，共通コードで定義した変数`random_seed`に格納されている値を乱数の種として用いる
* 9行目: `fit`メソッドで分類木モデルの学習を実行

In [ ]:
# モデルの学習
df = pd.read_csv('KvsT.csv')
x_cols = ['身長', '体重', '年代']
t_col = ['派閥']
x = df[x_cols]
t = df[t_col]
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(random_state=random_seed)
model.fit(X=x, y=t)
print('学習完了！')

### 学習済みモデルを使った予測
* 目的変数が未知の説明変数から目的変数の値を推測することを予測と呼ぶ
* 今回の例は分類に対する予測となる
* 学習済みモデルが得られると，それを使って予測ができる
* 予測は`predict`メソッドを用いる
  * 書式: `変数.predict(X=新たな説明変数)`
  * 「`X=`」で説明変数を表すDataFrameやリストを指定
  * 戻り値は予測結果（この例の場合は派閥）
  * 予測結果のデータ型はNumPyの配列（ndarray）
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: 説明変数の列名を要素とするリスト`['身長', '体重', '年代']`を変数`x_cols`に代入
* 4行目: 目的変数の列名（リスト）`['rent_price']`を変数`t_col`に代入
* 5行目: DataFrame `df` から,`df[x_cols]`で，説明変数の列だけ取り出し変数`x`に代入
* 6行目: DataFrame `df` から,`df[t_col]`で目的変数の列だけ取り出し変数`t`に代入
* 7行目: `sklearn` (scikit-learn) の`tree`モジュール内の`DecisionTreeClassifier`クラスをインポート
* 8行目: `DecisionTreeClassifier(random_state=random_seed)`で，分類木モデルの学習を行うためのオブジェクトを`DecisionTreeClassifier`クラスから生成し，変数`model`に代入
  * `random_state`で乱数の種（シード）を指定する
  * ここでは，共通コードで定義した変数`random_seed`に格納されている値を乱数の種として用いる
* 9行目: `fit`メソッドで分類木モデルの学習を実行
* 10行目: 新しい3人分の説明変数（リスト`[身長, 体重, 年代]`）のリスト`[[170, 70, 20], [160, 50, 10], [165, 75, 30]]`を変数`newdata`に代入
* 11行目: `predict`メソッドを用いて，新たな説明変数`newdata`に対する予測を行い，その結果（`predict`メソッドの戻り値）を`print`関数で表示
  * `f'予測結果： {model.predict(X=newdata)}'` は f-string

In [ ]:
# 学習済みモデルによる予測
df = pd.read_csv('KvsT.csv')
x_cols = ['身長', '体重', '年代']
t_col = ['派閥']
x = df[x_cols]
t = df[t_col]
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier(random_state=random_seed)
model.fit(X=x, y=t)
newdata = [[170, 70, 20], [160, 50, 10], [165, 75, 30]]
print(f'予測結果： {model.predict(X=newdata)}')

### 実習
上のコードと同様にして，分類木モデルの学習をした後，学習済みモデルを使って，身長「155」，体重「45」，年代「20」に対する予測結果を表示するコードを作成・実行せよ．

**［実行結果］**
```
予測結果： ['きのこ']
```

### 学習済みモデルの描画
* 分類木モデルは木構造で視覚的に表現できる
* 学習済みの分類木モデルの描画は`tree`モジュールの`plot_tree`関数を使う（インポートする必要がある）
* このとき，`matplotlib`の`pyplot`モジュールも利用する
* `plot_tree`関数の書式: `plot_tree(decision_tree=オブジェクト, feature_names=説明変数名)`
  * 「`decision_tree=`」で学習済みモデルの情報を持つオブジェクト（変数）を指定
  * 「`feature_names=`」で図中に表示する説明変数の名前を要素とするリストを指定
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: 説明変数の列名を要素とするリスト`['身長', '体重', '年代']`を変数`x_cols`に代入
* 4行目: 目的変数の列名（リスト）`['rent_price']`を変数`t_col`に代入
* 5行目: DataFrame `df` から,`df[x_cols]`で，説明変数の列だけ取り出し変数`x`に代入
* 6行目: DataFrame `df` から,`df[t_col]`で目的変数の列だけ取り出し変数`t`に代入
* 7行目: `sklearn` (scikit-learn) の`tree`モジュール内の`DecisionTreeClassifier`クラス，及び`plot_tree`関数をインポート
* 8行目: `DecisionTreeClassifier(random_state=random_seed)`で，分類木モデルの学習を行うためのオブジェクトを`DecisionTreeClassifier`クラスから生成し，変数`model`に代入
  * `random_state`で乱数の種（シード）を指定する
  * ここでは，共通コードで定義した変数`random_seed`に格納されている値を乱数の種として用いる
* 9行目: `fit`メソッドで分類木モデルの学習を実行
* 10行目: `pyplot`モジュールの`figure`関数を使って，図のサイズ（高さと幅）を指定
  * 引数「`figsize=(幅, 高さ)`」でサイズを指定
  * 単位は「インチ（inch）」 
* 11行目: `plot_tree`関数で分類木を描画
  * 引数「`decision_tree=model`」で描画する分類木モデルを指定
  * 引数「`feature_names=x_cols`」で図中に表示する説明変数名（`['身長', '体重', '年代']`）を指定
* 12行目: `show`関数で，それまでに設定した図（分類木モデル）を実行画面に表示


In [ ]:
# 学習済みモデルの可視化
df = pd.read_csv('KvsT.csv')
x_cols = ['身長', '体重', '年代']
t_col = ['派閥']
x = df[x_cols]
t = df[t_col]
from sklearn.tree import DecisionTreeClassifier, plot_tree
model = DecisionTreeClassifier(random_state=random_seed)
model.fit(X=x, y=t)
plt.figure(figsize=(10, 12))
plot_tree(decision_tree=model, feature_names=x_cols)
plt.show()

### 描画した分類木の見方
* 最上部のノードから出発して，各ノードの分岐条件によって左下または右下のノードへ進んでいく
* 条件を満たしているデータは左下のノードへ，そうでなければ右下のノードへ進む
* 葉ノードに達したらvalueの値が最大のものを予測値とする
  * 左が「きのこ」で右が「たけのこ」
* 例えば，身長170, 体重70, 年代20 の場合は，下図のように辿って，予測結果が「きのこ」になる
* 詳細は次回以降に説明する

  <img src="./fig/classification_tree_sample.jpg" width="600">

### モデルの評価
* 学習したモデルが必ず正しい予測結果を返すとは限らない
* 選択したモデルや利用したデータが不適切だった場合には，誤った学習をしてしまって，予測結果と実際の値に大きなずれが生じる
* そのため，モデルの予測性能を評価する必要がある
* モデルによって具体的な評価指標は異なる
* **精度**（Accuracy）は，教師あり学習における代表的なモデル評価指標の一つ
* 精度は正解率とも呼ばれるが，機械学習の文脈では「精度（Accuracy）」のほうが一般的
* ［精度］＝［実際の値（目的変数）と予測結果が一致しているデータ数］÷［総データ数］
* 精度は`DecisionTreeClassifier`クラスの`score`メソッドで計算できる
* 書式: `変数.score(X=説明変数, y=目的変数)`
  * `変数`は`DecisionTreeClassifier`クラスのオブジェクト
  * 学習に利用していない別のデータを使ってもよい（後述）
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: 説明変数の列名を要素とするリスト`['身長', '体重', '年代']`を変数`x_cols`に代入
* 4行目: 目的変数の列名（リスト）`['rent_price']`を変数`t_col`に代入
* 5行目: DataFrame `df` から,`df[x_cols]`で，説明変数の列だけ取り出し変数`x`に代入
* 6行目: DataFrame `df` から,`df[t_col]`で目的変数の列だけ取り出し変数`t`に代入
* 7行目: `sklearn` (scikit-learn) の`tree`モジュール内の`DecisionTreeClassifier`クラスをインポート
* 8行目: `DecisionTreeClassifier(random_state=random_seed)`で，分類木モデルの学習を行うためのオブジェクトを`DecisionTreeClassifier`クラスから生成し，変数`model`に代入
  * `random_state`で乱数の種（シード）を指定する
  * ここでは，共通コードで定義した変数`random_seed`に格納されている値を乱数の種として用いる
* 9行目: `fit`メソッドで分類木モデルの学習を実行
* 10行目: `model.score(X=x, y=t)`で精度（Accuracy）を計算し，`print`関数でその結果を表示
  * `f'精度（Accuracy）: {model.score(X = x, y = t)}'`は，f-string

In [ ]:
# モデルの評価
df = pd.read_csv('KvsT.csv')
x_cols = ['身長', '体重', '年代']
t_col = ['派閥']
x = df[x_cols]
t = df[t_col]
from sklearn import tree
model = tree.DecisionTreeClassifier(random_state=random_seed)
model.fit(X=x, y=t)
print(f'精度（Accuracy）: {model.score(X=x, y=t)}')

# 分類木モデルの学習のしくみ

## 分類木モデルの学習
* 分類のためにデータの各説明変数（特徴量）に対して，条件分岐を繰り返すモデル
* モデルの構造が視覚的に表現できるので，「モデルを解釈しやすい」という特徴がある
* 「どの特徴量を，どの値で分けるか」という分岐ルール（下図のようなイメージ）がパラメータに相当する
* この分岐ルールをデータから定めることが，分類木モデルの学習となる
* 分岐ルールは，うまく分類できるように決めていく ⇒ **不純度**を用いる

  <img src="./fig/classification_tree_sample1.jpg" width="400">

## 不純度に基づく分岐ルールの選択
* 不純度： 選択した条件による分類の「良さ」の基準 ⇒ 分岐後のデータに対する正解の割合
* 小さいほど良い（うまく分類できている）
* 分類木モデルの学習では，様々な条件を考えて，親ノードの不純度から，子ノードの不純度がどう変化したかを知らべる
* 代表的な不純度の指標（詳細は省略）
  * ジニ係数
  * エントロピー

  <img src="./fig/classification_tree_nodes.jpg" width="250">


## 不純度のイメージ
* 下図は，4つのデータを3パターンの分岐ルールで分類している
* ルール1：不純度が非常に小さい
* ルール2：不純度が非常に大きい
* ルール3：左の子ノードは不純度がやや大きいが，右の子ノードは非常に小さい

  <img src="./fig/classification_tree_impurity1.jpg" width="650">

* 分割したデータに対して，さらに分岐ルール（ルール4）を加えることで不純度の小さい子ノードが作れる

  <img src="./fig/classification_tree_impurity2.jpg" width="650">

## 学習のしくみの概要
* 学習の流れ（イメージ）
  * 親ノードにおいて考えられる様々な分岐ルールと，それによる分岐後の子ノードの不純度を計算
  * その中で不純度が最も小さくなる条件を選択
  * 分岐が進まなくなるまで繰り返す
* まともに学習する（考えられるすべての分岐ルールを調べる）と大変 ⇒ 条件をランダムに絞り込んで調べることにする（手抜きする）
  * 条件をランダムに絞り込むために乱数を使用する
  * ただし，再現性を確保するために乱数の種（random_state）は一般に固定して学習
* さらに，分類木の最大深さ（`max_depth`）も学習する際に指定しておく


# アヤメデータを使った分類木モデルの学習

## データの読み込みと確認
* 例として，csvファイル「iris.csv」を用いる
  * 出典: 須藤秋良, 株式会社フレアリンク: スッキリわかるPythonによる機械学習入門, インプレス, 2020
* このデータは，1936年にRonald A. Fisherが発表した判別分析に関する論文で用いられているデータ（以降，アヤメデータと呼ぶ）がもとになっている
  * オリジナルのデータのURL: https://archive.ics.uci.edu/ml/datasets/iris
* 今回使用するデータは，このオリジナルのデータを加工したもの
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `display`関数を使ってデータ（`df`の内容）を表示


In [ ]:
# データの読み込み
df = pd.read_csv('iris.csv')
display(df)

## アヤメの種類の確認
* ここでは，がく片長さ，がく片幅，花弁長さ，花弁幅を説明変数，種類を目的変数（正解データ）として，がく片長さ，がく片幅，花弁長さ，花弁幅から花の種類を分類することを考えていく 
* まずは，目的変数が何種類で何個ずつあるか，以下のコードで確認する
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: 「種類」の値の頻度を抽出し，その結果を`display`関数で表示
>* `df['種類'].value_counts()`で，DataFrameである`df`の列「種類」にどんな値がそれぞれ何個あるのかを調べる
>* 一般的な書式: `df['列名'].value_counts()`
>* `df`は任意のDataFrame
>* `value_counts`メソッドの戻り値は Series
>* 各値がSeriesのindexで，そのデータ数がSeriesの値となる
>* このSeriesのデータを，pandasの`DataFrame`関数でDataFrameに変換 ⇒ `pd.DataFrame(df['種類'].value_counts())`
>* 変換したDataFrameを`display`関数で表示

In [ ]:
# アヤメの種類の確認
df = pd.read_csv('iris.csv')
display(pd.DataFrame(df['種類'].value_counts()))

* 上記コードによって，花の種類（目的変数）は，以下の3種であることが確認できた．
>*   setosa（セトサ／ヒオウギアヤメ）
>*   versicolor（バージカラー／ブルーフラッグ）
>*   virginica（バージニカ／バージニアアイリス）  
*  また，各種のデータが50個ずつあることも確認できた．

## 説明変数の各種統計量
* 次に，4つの説明変数（がく片長さ～花弁幅）の各種統計量を`describe`メソッドで求める
* 「種類」の列は数値ではないので，除外される

**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `describe`メソッドで，4つの説明変数（がく片長さ～花弁幅）の各種統計量を計算し，その結果を`display`関数で表示

In [ ]:
# 説明変数の統計量の確認
df = pd.read_csv('iris.csv')
display(df.describe())

## データの前処理

### 欠損値の確認
* データの前処理として，アヤメデータに欠損値がないかを確認する
* 欠損値の確認には，`isnull`メソッドを使う
* `isnull`メソッドを使うと，DataFrame内の欠損値（NaN: Not a Number）がある場所は「`True`」，そうでない場所は「`False`」となるDataFrameが作成できる
* 詳細は教科書第5章，「[ノートブック：SeriesとDataFrameの基本操作](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook_se_df.ipynb)」を参照
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `isnull`メソッドで`df`の中に欠損値があるか確認し，その結果を`display`関数で表示
* 結果からインデックス「147」の「花弁長さ」に欠損があるとわかる（すべてのデータが見えてないことに注意）

In [ ]:
# 欠損値の確認
df = pd.read_csv('iris.csv')
display(df.isnull())

* 次に，DataFrame全体で，どの列に欠損値があるかを`any`メソッドで確認する
* `any`メソッドを使うと，各列（or 各行）において，1つ以上「`True`」があるかを確認できる
  * 列（or 行）内に一つでも「`True`」があれば「`True`」を返す
* 引数`axis`で行方向（`= 0`）に見るか，列方向（`= 1`）に見るかを決める
  * 行方向は各列に対して上から下に見る
  * 列方向は各行に対して左から右に見る
* 戻り値は Series
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `isnull`メソッドと`any`メソッドで，`df`の中で欠損値がある列を確認し，その結果を`display`関数で表示
  * `df.isnull()`で，`df`内の欠損値を`True`，それ以外を`False`とした DataFrame に変換する（戻り値が DataFrame）
  * `df.isnull()`の戻り値に対して，`any`メソッドを使って`df.isnull().any(axis = 0)`とし，`True`（欠損値）がある列を`True`，ない列を`Flase`とした Series に変換する（戻り値が Series）
  * それを`DataFrame`関数でDataFrameに変換し，`display`関数で表示
  * `DataFrame`関数の引数`columns = ['欠損値']`で列名を「欠損値」と指定
*  実行結果から4つの説明変数に欠損値があることが確認できる

In [ ]:
# 欠損値の有無の確認
df = pd.read_csv('iris.csv')
display(pd.DataFrame(df.isnull().any(axis = 0), columns = ['欠損値']))

### 欠損値への対処

#### 欠損値のある行データを削除
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* `dropna`メソッドは，欠損値のあるデータを削除したDataFrameを戻り値として返す
* 引数`how`と`axis`については以下のとおり
  * `how`: 「`'any'`」とすると欠損値が一つでもあれば削除，「`'all'`」とすると全てが欠損値であれば削除
  * `axis`: 「1」とすると欠損値のある列を削除，「0」とすると行を削除
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `df.dropna(how='any', axis=0)`で，`df`に対して一つ以上の欠損値がある行を削除し，変数`df`に代入
* 4行目: `display`関数で，`df`の内容を表示
* 削除した結果，150行から143行になる

In [ ]:
# 欠損値のある行を削除
df = pd.read_csv('iris.csv')
df = df.dropna(how = 'any', axis = 0)
display(df)

*  念のため，`df`に欠損値がなくなったを次のコードで確認する
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `df.dropna(how='any', axis=0)`で，`df`に対して一つ以上の欠損値がある行を削除し，変数`df`に代入
* 4行目: `isnull`メソッドと`any`メソッドで，`df`の中で欠損値がある列を確認し，その結果を`display`関数で表示
  * `df.isnull()`で，`df`内の欠損値を`True`，それ以外を`False`とした DataFrame に変換する（戻り値が DataFrame）
  * `df.isnull()`の戻り値に対して，`any`メソッドを使って`df.isnull().any(axis=0)`とし，`True`（欠損値）がある列を`True`，ない列を`Flase`とした Series に変換する（戻り値が Series）
  * それを`DataFrame`関数でDataFrameに変換し，`display`関数で表示
  * `DataFrame`関数の引数`columns=['欠損値']`で列名を「欠損値」と指定

In [ ]:
# 欠損値のある行を削除した後の欠損値の有無の確認
df = pd.read_csv('iris.csv')
df = df.dropna(how = 'any', axis = 0)
display(pd.DataFrame(df.isnull().any(axis = 0), columns = ['欠損値']))

#### 欠損値を算術平均に置き換える
* 欠損値のあるデータを削除するのではなく，該当する説明変数の代表値（算術平均や中央値）に置き換えて対処することもある（こちらのほうが一般的）
* ここでは，算術平均を使って穴埋めする
* 欠損値を別の値に置き換えるには，`fillna`メソッドを使う
* 書式: `df.fillna(置き換える値)`
* 列ごとに置き換える値を変える場合は，Seriesを引数として指定する
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `fillna`メソッドを使って，`df.fillna(df[['がく片長さ','がく片幅', '花弁長さ', '花弁幅']].mean())`とし，欠損値を各説明変数の算術平均にそれぞれ置き換えて変数`df`に代入
  * `fillna`メソッドの引数は，各説明変数の算術平均
  * 各説明変数の算術平均は，`mean`メソッドを使って `df[['がく片長さ','がく片幅', '花弁長さ', '花弁幅']].mean()`とすることで計算できる
  * `df[['がく片長さ', 'がく片幅', '花弁長さ', '花弁幅']].mean()`は，Seriesとなる
  * `df.mean()`とすると，「種類」の算術平均も計算することになるが，この列の値は数値ではないため計算できない ⇒ エラーとなる
* 4行目: `df`の内容を`display`関数で表示
* ここでのモデルの学習には，欠損値を算術平均に置き換えたデータを使用していく

In [ ]:
# 欠損値を平均値で補完
df = pd.read_csv('iris.csv')
df = df.fillna(df[['がく片長さ', 'がく片幅', '花弁長さ', '花弁幅']].mean())
display(df)

## データの分割

### ホールドアウト法
* 派閥データを使った分類木の学習と評価では，学習に使用したデータ（訓練データと呼ぶ）を使って精度（Accuracy）を計算していた
* しかし，機械学習は訓練データに当てはまるようにモデルを学習しているので必然的に精度は高くなる
* よって，一般的には，訓練データではない別のデータ（テストデータと呼ぶ）を使ってモデルの評価を行っている
* ここでは，代表的なモデルの評価方法の一つである**ホールドアウト法**を使ってモデルを評価する
* そのために，まず，説明変数と目的変数を訓練データとテストデータの2つに分割する
* 分割の割合は，説明変数と目的変数で同じとする

  <img src="./fig/holdout_division_data.jpg" width="500">

* モデルの学習には，訓練データのみを使う

  <img src="./fig/holdout_learning.jpg" width="500">

* テストデータは，学習したモデルを評価するときに使う
* 例えば，分類木であれば精度を計算するときにテストデータを使う

  <img src="./fig/holdout_evaluation.jpg" width="500">

### 分割方法
* 訓練データとテストデータの分割は，`sklearn`（scikit-learn）の`model_selection`モジュールの中にある`train_test_split`関数を使う
* `train_test_split`関数のインポートの書式: `from sklearn.model_selection import train_test_split`
* `train_test_split`関数の引数
  * 1番目の引数: 分割する説明変数
  * 2番目の引数: 分割する目的変数
  * `test_size`: 分割するテストデータの割合（サイズ）
  * `random_state`: 分割するデータはランダムに割り振られるので乱数の種を指定
* `train_test_split`関数の戻り値は分割した4つのデータのリストとなる
  * 訓練データの説明変数
  * テストデータの説明変数
  * 訓練データの目的変数
  * テストデータの目的変数

### 実習
**［実習内容］**
* 以下の「以下のコードの処理内容」に従って，コードを完成させる
* 3～7行目に適切なコードを記述する
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* **3行目**: `fillna`メソッドを使って，`df.fillna(df[['がく片長さ','がく片幅', '花弁長さ', '花弁幅']].mean())`とし，欠損値を各説明変数の算術平均にそれぞれ置き換えて変数`df`に代入
* **4行目**: 説明変数の列名を要素とするリスト`['がく片長さ', 'がく片幅', '花弁長さ', '花弁幅']`を変数`x_cols`に代入
* **5行目**: 目的変数の列名（リスト）`['種類']`を変数`t_col`に代入
* **6行目**: DataFrame `df` から,`df[x_cols]`で，説明変数の列だけ取り出し変数`x`に代入
* **7行目**: DataFrame `df` から,`df[t_col]`で目的変数の列だけ取り出し変数`t`に代入
* 8行目: `model_selection`モジュールの`train_test_split`関数のインポート
* 9行目: `train_test_split`関数を使って説明変数`x`と目的変数`t`を訓練データとテストデータにそれぞれ分割
  * `test_size=0.3`として，訓練データを7割（105個），テストデータを3割（45個）に分割
  * `random_state=random_seed`として，乱数を固定する ⇒ 結果が同じになる
  * `x_train`: 訓練データの説明変数
  * `x_test`: テストデータの説明変数
  * `t_train`: 訓練データの目的変数
  * `t_test`: テストデータの目的変数
* 10行目: `print`関数で区切り線「================ x_train ================」を表示
* 11行目: `display`関数で変数`x_train`（訓練データの説明変数）の先頭2行を表示
* 12行目: `print`関数で区切り線「================ t_train ================」を表示
* 13行目: `display`関数で変数`t_train`（訓練データの目的変数）の先頭2行を表示
* 14行目: `print`関数で区切り線「================ x_testn ================」を表示
* 15行目: `display`関数で変数`x_test`（テストデータの説明変数）の先頭2行を表示
* 16行目: `print`関数で区切り線「================ t_test ================」を表示
* 17行目: `display`関数で変数`t_test`（テストデータの目的変数）の先頭2行を表示

**［実行結果］**

  <img src="./fig/exercise_iris_Data_splitting.jpg" width="320">

In [ ]:
# データの分割
df = pd.read_csv('iris.csv')





from sklearn.model_selection import train_test_split
x_train, x_test, t_train, t_test = train_test_split(x, t, test_size=0.3, random_state=random_seed)
print('================ x_train ================')
display(x_train.head(2))
print('================ t_train ================')
display(t_train.head(2))
print('================ x_test ================')
display(x_test.head(2))
print('================ t_test ================')
display(t_test.head(2))

## 分類木モデルの学習
* 派閥データと同様にして，分類木モデルの学習と予測を行う

### 分類木モデルの学習準備
* `sklearn`の`tree`モジュールの`DecisionTreeClassifier`クラスをインポート ⇒ `from sklearn.tree import DecisionTreeClassifier`
* オブジェクトを生成して変数に代入 ⇒ `変数名 = DecisionTreeClassifier(max_depth=[木の最大深さ], random_state=random_seed)`
* 引数`max_depth`で，木の最大深さ（整数値）を指定できる

### 分類木モデルの学習
* 分類木モデルの学習は，`fit`メソッドを利用して実行する
* ホールドアウト法において，`fit`メソッドの引数には，訓練データの説明変数と目的変数データを指定する
* モデルの学習の書式: `変数.fit(X=訓練データの説明変数, y=[訓練データの目的変数])`

### 分類木モデルの予測
* 学習済みモデルが得られると，`predict`メソッドを使って予測ができる
* 書式: `変数名.predict(X=新たな説明変数)`

### 分類木モデルの評価
* 派閥データと同様に，精度（Accuracy）を用いて評価する
* ［精度］＝［実際の値（目的変数データ）と予測結果が一致しているデータ数］÷［総データ数］
* 精度は`DecisionTreeClassifier`クラスの`score`メソッドで計算できる
* 書式: `変数.score(X=説明変数, y=目的変数)`
* ホールドアウト法において，精度はテストデータを用いて計算する
* 比較・参考として訓練データに対する精度を計算することもある

### 分類木モデルの描画
* 派閥データと同様にして，`tree`モジュールの`plot_tree`関数で分類木モデルを描画する
* `plot_tree`関数の書式: `tree.plot_tree(decision_tree=オブジェクト, feature_names=説明変数名)`
  * 「`decision_tree=`」で学習済みモデルの情報を持つオブジェクト名を指定
  * 「`feature_names=`」で図中に表示する説明変数名のリストを指定

## 実習：分類木モデルの学習
**［実習内容］**
* 以下の「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する

**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `fillna`メソッドを使って，`df.fillna(df[['がく片長さ','がく片幅', '花弁長さ', '花弁幅']].mean())`とし，欠損値を各説明変数の算術平均にそれぞれ置き換えて変数`df`に代入
* 4行目: 説明変数の列名を要素とするリスト`['がく片長さ', 'がく片幅', '花弁長さ', '花弁幅']`を変数`x_cols`に代入
* 5行目: 目的変数の列名（リスト）`['種類']`を変数`t_col`に代入
* 6行目: DataFrame `df` から,`df[x_cols]`で，説明変数の列だけ取り出し変数`x`に代入
* 7行目: DataFrame `df` から,`df[t_col]`で目的変数の列だけ取り出し変数`t`に代入
* 8行目: `model_selection`モジュールの`train_test_split`関数のインポート
* 9行目: `train_test_split`関数を使って説明変数`x`と目的変数`t`を訓練データとテストデータにそれぞれ分割
* 10行目: `sklearn` (scikit-learn) の`tree`モジュールの`DecisionTreeClassifier`クラスをインポート
* 11行目: 分類木モデルの学習を行うためのオブジェクトを`DecisionTreeClassifier`クラスから生成し，変数`model_tree`に代入
  * `max_depth = 2`で最大深さを2とする
  * `random_state = random_seed`で乱数の種（シード）を指定する
* 12行目: `fit`メソッドで分類木モデルの学習を実行
  * 学習には訓練データ`x_train`, `t_train`を使う
* 13行目: 新しい2つの説明変数のリスト`[[0.5, 0.8, 0.6, 0.6], [0.2, 0.7, 0.2, 0.2]]`を変数`newdata`に代入
* 14行目: `predict`メソッドを用いて，新たな説明変数 `newdata`に対する予測を行い，その結果（`predict`メソッドの戻り値）を`print`関数で表示
  * `f'予測結果： {model_tree.predict(X = newdata)}'`は f-string
* 15行目: `score`メソッドで，**訓練データ**に対する精度を計算して変数`score_train`に代入
* 16行目: `score`メソッドで，**テストデータ**に対する精度を計算して変数`score_test`に代入
* 17行目: `print`関数とf-stringを使って，訓練データに対する精度（`score_train`）とテストデータに対する精度（`score_test`）を小数点以下3桁まで表示

**［実行結果］**
```
予測結果： ['versicolor' 'setosa']
訓練データの精度=0.933 / テストデータの精度=0.956
```

In [ ]:
# 分類木モデルの学習
df = pd.read_csv('iris.csv')





from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier


newdata = [[0.5, 0.8, 0.6, 0.6], [0.2, 0.7, 0.2, 0.2]]
print(f'予測結果： {model_tree.predict(newdata)}')


print(f'訓練データの精度={score_train:.3f} / テストデータの精度={score_test:.3f}')

## 実習：分類木モデルの描画
**［実習内容］**
* 以下の「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する

**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `fillna`メソッドを使って，`df.fillna(df[['がく片長さ','がく片幅', '花弁長さ', '花弁幅']].mean())`とし，欠損値を各説明変数の算術平均にそれぞれ置き換えて変数`df`に代入
* 4行目: 説明変数の列名を要素とするリスト`['がく片長さ', 'がく片幅', '花弁長さ', '花弁幅']`を変数`x_cols`に代入
* 5行目: 目的変数の列名（リスト）`['種類']`を変数`t_col`に代入
* 6行目: DataFrame `df` から,`df[x_cols]`で，説明変数の列だけ取り出し変数`x`に代入
* 7行目: DataFrame `df` から,`df[t_col]`で目的変数の列だけ取り出し変数`t`に代入
* 8行目: `model_selection`モジュールの`train_test_split`関数のインポート
* 9行目: `train_test_split`関数を使って説明変数`x`と目的変数`t`を訓練データとテストデータにそれぞれ分割
* 10行目: `sklearn` (scikit-learn) の`tree`モジュール内の`DecisionTreeClassifier`クラス，及び`plot_tree`関数をインポート
* 11行目: 分類木モデルの学習を行うためのオブジェクトを`DecisionTreeClassifier`クラスから生成し，変数`model_tree`に代入
  * `max_depth = 2`で最大深さを2とする
  * `random_state = random_seed`で乱数の種（シード）を指定する
* 12行目: `fit`メソッドで分類木モデルの学習を実行
  * 学習には訓練データ`x_train`, `t_train`を使う
* 13行目: `pyplot`モジュールの`figure`関数を使って，図のサイズ（高さと幅）を `(5, 5)` に設定
* 14行目: `plot_tree`関数で分類木を描画
* 15行目: `show`関数で，それまでに設定した図（分類木モデル）を実行画面に表示

**［実行結果］**

  <img src="./fig/exercise_iris_classification_tree.jpg" width="320">

In [ ]:
# 分類木モデルの描画
df = pd.read_csv('iris.csv')





from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier, plot_tree


plt.figure(figsize=(5, 5))

plt.show()